# YouTube Comment Toxicity Analysis
**Query:** `Wahlbetrug Deutschland` (Election Fraud Germany)  
**Model:** `EIStakovskii/german_toxicity_classifier_plus_v2`  
**Platform:** YouTube Data API v3

## 1. Setup & Authentication

In [ ]:
from googleapiclient.discovery import build
import pandas as pd

# Set your YouTube Data API v3 key here
# Obtain one at: https://console.cloud.google.com/
api_key = "YOUR_YOUTUBE_API_KEY"
youtube = build('youtube', 'v3', developerKey=api_key)

## 2. Video Search

In [ ]:
# Search for videos matching the query, filtered to Germany
request = youtube.search().list(
    q="Wahlbetrug Deutschland",
    part="snippet",
    maxResults=10,
    regionCode="DE",
    relevanceLanguage="de"
)
results = request.execute()

# Extract video IDs from search results
video_ids = [item['id']['videoId']
             for item in results['items']
             if item['id']['kind'] == 'youtube#video']

print(f"Videos found: {len(video_ids)}")
print(video_ids)

Videos found: 10
['dUag-RGvDlQ', 'CK2d_O-4xm8', 'ylKRZxLludA', 'Ws7ZZM_mD-M', 'InGw3VtLx4A', 'i6xlqV42vRQ', '6mfLZCVnluk', 'pyXS4ytSIGY', '1OY0j3cNmmY', '1-mjJCbVDtg']


## 3. Comment Collection

In [ ]:
comments = []

for video_id in video_ids:
    try:
        response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText"
        ).execute()

        for item in response['items']:
            comment = item['snippet']['topLevelComment']['snippet']
            comments.append({
                'video_id': video_id,
                'text': comment['textDisplay'],
                'likes': comment['likeCount'],
                'date': comment['publishedAt']
            })
    except:
        pass  # Skip videos with comments disabled

df = pd.DataFrame(comments)
print(f"Total comments collected: {len(df)}")
df.head()

Total comments collected: 293


,video_id,text,likes,date
0,dUag-RGvDlQ,Ja das können die grünen lügen und betrügen si...,0,2026-03-09T19:51:31Z
1,dUag-RGvDlQ,Özi brigt den fast gratissrom.\nAlle länder wo...,0,2026-03-09T19:23:51Z
2,dUag-RGvDlQ,"Wie meinen Sie, wer tut noch grün wählen😢😢😢",0,2026-03-09T19:22:40Z
3,dUag-RGvDlQ,Harbeck und Bärbock weg \nUnd jetzt kommt der...,0,2026-03-09T18:34:29Z
4,dUag-RGvDlQ,Für die Verlierer ist das IMMER Wahlbetrug! ...,0,2026-03-09T18:33:00Z


## 4. Toxicity Classification
Model: [`EIStakovskii/german_toxicity_classifier_plus_v2`](https://huggingface.co/EIStakovskii/german_toxicity_classifier_plus_v2)

In [ ]:
from transformers import pipeline

# Load the German toxicity classification model
classifier = pipeline("text-classification",
                      model="EIStakovskii/german_toxicity_classifier_plus_v2")

# Classify each comment (truncate to 512 tokens for BERT input limit)
df['toxicity'] = df['text'].apply(
    lambda x: classifier(x[:512])[0]['label']
)

df['toxicity'].value_counts()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: EIStakovskii/german_toxicity_classifier_plus_v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,count
toxicity,
toxic,236
neutral,57


## 5. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Toxicity distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
df['toxicity'].value_counts().plot(kind='bar',
                                    color=['salmon', 'lightgreen'])
plt.title("Toxicity Distribution\n(Wahlbetrug Comments)")
plt.xticks(rotation=0)
plt.ylabel("Count")

# 2. Temporal trend of toxic comments
plt.subplot(1, 2, 2)
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M')
df[df['toxicity']=='toxic'].groupby('month').size().plot(kind='line',
                                                          color='red',
                                                          marker='o')
plt.title("Toxic Comments Over Time")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

/tmp/ipykernel_181/1047226580.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['date'].dt.to_period('M')


## 6. Export

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>